Now project all cells to the latent space and visualize the results using UMAP.
Li_2025_NatCa is an epithelial-only query (CRC and PM), excluded from reference HVG selection and training. Map all its cells in the query step below.
the scANVI embedding will be thrown away

In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from pathlib import Path
import scvi
import anndata as ad
import sys

sys.path.insert(0, str(Path.home() / "project/crc-pm/code"))
import src


In [ ]:
MODEL_DIR = Path.home() / "project/crc-pm/result/scvi/scvi_model"
model_scvi = scvi.model.SCVI.load(MODEL_DIR)
DATA_IN_DIR = Path.home() / "project/crc-pm/result/label-transfer/qc_and_transferred.h5ad"
adata_full = sc.read_h5ad(DATA_IN_DIR)

In [ ]:
HVG_gene = pd.read_csv(
    str(Path.home() / "project/crc-pm/result/hvg" / "hvg_flags.tsv"),
      sep="\t",
      index_col="gene_id",
  )
adata_full.var['highly_variable'] = HVG_gene
adata_full.obs["Study_Assay"] = (
      adata_full.obs["Study"].astype(str)
      + "_"
      + adata_full.obs["Assay"].astype(str)
  ).astype("category")

In [ ]:
# 从模型 registry 恢复训练时使用的 HVG 及其顺序
# 模型保存时没有保存 AnnData，因此不能使用 model_scvi.adata.var_names
hvg_genes = pd.Index(model_scvi.get_var_names())

assert hvg_genes.is_unique
assert hvg_genes.isin(adata_full.var_names).all()


# ============================================================
# 1. 直接映射模型训练时见过的 batch
# ============================================================

known_studies = [
    "Berlin_2023_JCliInv",
    "Lenos_2022_NatCom",
    "Pelka_2021_Cell",
    "Lee_2020_NatGen_KUL3",
    "Lee_2020_NatGen_SMC",
]

known_mask = (
    adata_full.obs["Study"]
    .astype(str)
    .isin(known_studies)
)

adata_known = adata_full[known_mask].copy()

# 重新生成 category，避免残留 Li_2025_NatCa 的未使用 category
adata_known.obs["Study_Assay"] = (
    adata_known.obs["Study"].astype(str)
    + "_"
    + adata_known.obs["Assay"].astype(str)
).astype("category")

adata_known_hvg = adata_known[:, hvg_genes].copy()

latent_known = model_scvi.get_latent_representation(
    adata=adata_known_hvg,
    batch_size=512,
)


# ============================================================
# 2. 将 Li_2025_NatCa 作为新 query 映射
# ============================================================

adata_query = adata_full[
    adata_full.obs["Study"].astype(str).eq("Li_2025_NatCa")
].copy()

adata_query.obs["Study_Assay"] = (
    adata_query.obs["Study"].astype(str)
    + "_"
    + adata_query.obs["Assay"].astype(str)
).astype("category")

adata_query_hvg = adata_query[:, hvg_genes].copy()

query_model = scvi.model.SCVI.load_query_data(
    adata=adata_query_hvg,
    reference_model=model_scvi,
)

query_model.train(
    max_epochs=100,
    batch_size=512,
    plan_kwargs={"weight_decay": 0.0},
)

latent_query = query_model.get_latent_representation(
    batch_size=512,
)


# ============================================================
# 3. 按 adata_full 原始细胞顺序合并 latent
# ============================================================

latent_full = np.full(
    (adata_full.n_obs, latent_known.shape[1]),
    np.nan,
    dtype=np.float32,
)

known_pos = adata_full.obs_names.get_indexer(
    adata_known.obs_names
)

query_pos = adata_full.obs_names.get_indexer(
    adata_query.obs_names
)

assert (known_pos >= 0).all()
assert (query_pos >= 0).all()

latent_full[known_pos] = latent_known
latent_full[query_pos] = latent_query

assert np.isfinite(latent_full).all()

adata_full.obsm["X_scVI"] = latent_full

print(adata_full)
print(adata_full.obsm["X_scVI"].shape)

In [ ]:
sc.pp.neighbors(
    adata_full,
    use_rep="X_scVI",
    n_neighbors=20,
    metric="cosine",
    random_state=42,
)

sc.tl.umap(
    adata_full,
    min_dist=0.25,
    random_state=2026,)

In [ ]:
sc.pl.umap(
    adata_full,
    color=[
        "Study",
        "Tissue",
    ],)

In [ ]:
sc.pl.umap(
    adata_full[adata_full.obs['Study'].isin(['Li_2025_NatCa','Berlin_2023_JCliInv','Lenos_2022_NatCom'])],
    color=[
        "Study",
        "Tissue",
    ],)

### Normalization

In [ ]:
adata_full.layers["log1p"] = adata_full.layers["counts"].copy()

sc.pp.normalize_total(
    adata_full,
    target_sum=1e4,
    layer="log1p",
)

sc.pp.log1p(
    adata_full,
    layer="log1p",
)

In [ ]:
adata_full.layers["log1p"]

In [ ]:
cmap = src.GenCmap.saturated()
sc.pl.umap(
    adata_full,
    color=["EPCAM", "PTPRC", "COL1A1"],
    gene_symbols="gene_name",
    layer="log1p",
    cmap=cmap,
    vmin="p1",
    vmax="p99",
    size=0.5,
)

In [ ]:
adata_full.write_h5ad(Path.home() / "project/crc-pm/result/dr-anno" / "ready_for_anno.h5ad")